In [1]:
# Di dalam Jupyter Notebook Anda
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip

# 1. Inisialisasi Spark Session Lokal
builder = SparkSession.builder \
    .appName("ClusterSparkTaxi") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.plugins", "org.apache.spark.sql.connect.SparkConnectPlugin") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.connect.grpc.binding.port", "15002")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Cluster Spark Connect berhasil menyala di port 15002!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 20:12:30 WARN Utils: Your hostname, sanju, resolves to a loopback address: 127.0.1.1; using 192.168.1.39 instead (on interface wlp0s20f3)
26/05/21 20:12:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/sanju/miniconda3/envs/basic_dev/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/sanju/.ivy2.5.2/cache
The jars for the packages stored in: /home/sanju/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-32a83c0e-cc9a-4dfc-a456-d45ae4a2e2b7;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	foun

Cluster Spark Connect berhasil menyala di port 15002!


In [35]:
import logging

def ingest(spark, file_path, output_path):
    df = spark.read.format("parquet").load(file_path)
    logging.info(f"Data ingested from {file_path}.")

    try:
        df.write.format("delta").mode("append").save(output_path)
        logging.info(f"Data successfully written to {output_path}.")
    except Exception as e:
        logging.error(f"Error writing data to {output_path}: {e}")
    return logging.info("Ingestion process completed.")


def transform(spark, file_path, output_path):
    from pyspark.sql.functions import col, lit, when, unix_timestamp

    df = spark.read.format("delta").load(file_path)
    logging.info(f"Data read from {file_path}")

    df = df.withColumn(
        "duration_hours",
        (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
    )

    rules = [
        # === 1. IDENTITAS & PENYEDIA ===
        (~col("VendorID").isin([1, 2, 6, 7]),                              "invalid_vendor"),
        (~col("store_and_fwd_flag").isin(["Y", "N"]),                     "invalid_store_and_fwd_flag"),

        # === 2. WAKTU PERJALANAN ===
        (col("tpep_pickup_datetime").isNull(),                              "null_pickup_datetime"),
        (col("tpep_dropoff_datetime").isNull(),                             "null_dropoff_datetime"),
        ((col("duration_hours") < 0),                                       "negative_duration"), 
        ((col("duration_hours") > 24),                                      "duration_exceeds_24h"),

        # === 3. DETAIL PERJALANAN ===
        ((col("trip_distance") < 0),                                        "negative_trip_distance"),
        ((col("trip_distance") >= 300),                                     "trip_distance_outlier"),
        ((col("passenger_count") < 0),                                      "negative_passenger_count"),
        ((col("passenger_count") > 6),                                      "passenger_count_exceeds_6"),
        (~col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99]),                 "invalid_ratecode"),

        # === 4. LOKASI WILAYAH ===
        ((col("PULocationID") <= 0),                                        "invalid_pu_location_zero"),
        ((col("PULocationID") > 265),                                        "invalid_pu_location_out_of_range"),
        ((col("DOLocationID") <= 0),                                        "invalid_do_location_zero"),
        ((col("DOLocationID") > 265),                                       "invalid_do_location_out_of_range"),

        # === 5. METODE PEMBAYARAN & BIAYA ===
        (~col("payment_type").isin([1, 2, 3, 4, 5, 6]),                   "invalid_payment_type"),

        ((col("fare_amount") <= 0),                                         "non_positive_fare"),
        ((col("fare_amount") >= 500),                                       "fare_outlier"),

        ((col("extra") < 0),                                                "negative_extra"),
        ((col("extra") > 10),                                               "extra_outlier"),

        ((col("mta_tax") < 0),                                              "negative_mta_tax"),
        (~col("mta_tax").isin([0, 0.5]),                                   "invalid_mta_tax"),

        ((col("tip_amount") < 0),                                           "negative_tip"),
        ((col("payment_type") == 2) & (col("tip_amount") != 0),           "cash_tip_anomaly"),

        ((col("tolls_amount") < 0),                                         "negative_tolls"),
        ((col("tolls_amount") > 100),                                       "tolls_outlier"),

        (~col("improvement_surcharge").isin([0, 0.3, 1.0]),               "invalid_improvement_surcharge"),

        ((col("congestion_surcharge") < 0),                                 "negative_congestion_surcharge"),
        ((col("congestion_surcharge") > 5),                                 "congestion_surcharge_outlier"),

        ((col("airport_fee") < 0),                                          "negative_airport_fee"),
        ((col("airport_fee") > 5.0),                                        "airport_fee_outlier"),
    ]

    condition_chain = lit("valid")
    for condition, label in rules:
        condition_chain = when(condition, lit(label)).otherwise(condition_chain)

    df = df.withColumn("status", condition_chain)

    valid_df = df.filter(col("status") == "valid")
    invalid_df = df.filter(col("status") != "valid")

    valid_df.coalesce(2)
    valid_df.write.format("delta").mode("append").save(output_path)
    logging.info(f"Valid records written to {output_path}.")

    invalid_path = "data/broken/yellow_tripdata_2025-03/"
    invalid_df.coalesce(1)
    invalid_df.write.format("delta").mode("append").save(invalid_path)
    logging.info(f"Invalid records written to {invalid_path}.")

    logging.info("Transformation completed successfully.")

In [40]:
file_path = "data/yellow_tripdata_2025-03.parquet"

In [37]:
output_bronze = "data/bronze/yellow_tripdata_2025-03"
bronze = ingest(spark, file_path, output_bronze)

In [38]:
transform(spark, output_bronze, "data/silver/yellow_tripdata_2025-03")

In [39]:
spark.stop()